From Cross-Sections to Dynamics
===============================

**Author:** Ethan Ligon



Two ways to get change out of survey data: build a panel from cohorts, or
use one that already exists.



## Reading



Deaton is in your `reading/` folder on the hub, or [PDF](https://documents.worldbank.org/curated/en/203811547671768139/pdf/133790-PUB.pdf).

-   Deaton (1985), "Panel data from time series of cross-sections"
-   Deaton, ch. 2 §2.7 and ch. 6
-   Abadie, Athey, Imbens & Wooldridge (2023), "When should you adjust
    standard errors for clustering?" *QJE* 138:1–35



## The problem



## Pseudo-panels



## Identification: age, cohort, and time



## A genuine panel: Uganda



## Inference



## Building a pseudo-panel



In [1]:
# Show tracebacks without the library's internal frames: the line that
# failed, and why.  Change Plain to Verbose if you ever want the rest.
%xmode Plain

# The library audits its own corpus on first read and reports what it finds
# --- implausible quantities, NaN index keys, a column that is wholly null in
# one wave --- at multi-paragraph length.  Those reports are a work queue for
# whoever maintains the data, not something the room can act on, and they bury
# the output they are attached to.  Silenced here by their own
# "Set LSMS_..._STRICT=1" signature, which is precise: every other warning,
# pandas deprecations included, still shows.  Delete these two lines to read
# them.
import warnings
warnings.filterwarnings("ignore",
                        message=r"(?s).*Set LSMS_[A-Z_]+=1 to make this fatal")

# The first call that builds a table may print "DVC unavailable ...
# falling back to manual aggregation".  That is about how the library
# fetches its raw files, not about your data; the numbers are the same,
# and it does not recur once the table is built.

import matplotlib as mpl
mpl.rcParams.update({'figure.dpi': 110, 'font.size': 11,
                     'lines.linewidth': 1.8})

import lsms_library as ll
import numpy as np, pandas as pd

ghana = ll.Country('GhanaLSS')
roster = ghana.household_roster()
sample = ghana.sample()
food = ghana.food_expenditures()

# Age of the head, by (t, i).  Relationship == 'Head' identifies the head.
heads = roster[roster.Relationship.str.strip().str.lower().eq('head')]
age = heads.groupby(['t', 'i']).Age.first().rename('age')
age.groupby('t').describe().round(1)

In [1]:
# Wave midpoint year, so we can turn age into a birth year
year = {w: int(w[:4]) + 1 for w in ghana.waves}

x = food.groupby(['t', 'i']).sum().squeeze().rename('c')
d = pd.concat([x, age], axis=1).dropna()
d['year'] = [year[t] for t in d.index.get_level_values('t')]
d['born'] = d.year - d.age
d = d[(d.age >= 25) & (d.age <= 70)]

# Five-year cohorts
d['cohort'] = (d.born // 5) * 5
d['logc'] = np.log(d.c.where(d.c > 0))

# `sample` is indexed (i, t); everything else here is (t, i).  reindex
# against a differently-ordered MultiIndex does not raise -- it returns all
# NaN, and the pseudo-panel below comes out empty with no error anywhere.
# Reorder explicitly, then check that the merge actually matched.
w_all = sample.weight.reorder_levels(['t', 'i']).sort_index()
d['w'] = w_all.reindex(d.index)
assert d.w.notna().mean() > 0.9, f"only {d.w.notna().mean():.1%} of weights matched"


# Weighted cell means, as ratios of sums.  groupby-apply returning a Series
# is fragile across pandas versions; this is not.
dd = d.dropna(subset=['logc', 'w']).reset_index()
grp = dd.groupby(['cohort', 't'])
cells = pd.DataFrame({
    'logc': (dd.w * dd.logc).groupby([dd.cohort, dd.t]).sum() / grp.w.sum(),
    'age':  (dd.w * dd.age).groupby([dd.cohort, dd.t]).sum() / grp.w.sum(),
    'n':    grp.size(),
})
cells.head(12)

In [1]:
# Deaton's rule of thumb: cells want n >= 100.  How many survive?
print(f"{len(cells)} cells; {(cells.n >= 100).mean():.3f} of them have n >= 100")
cells.n.describe().round(0)

## Lifecycle consumption



In [1]:
import matplotlib.pyplot as plt

big = cells[cells.n >= 100].reset_index()
fig, ax = plt.subplots(figsize=(6.5, 4))
for c, g in big.groupby('cohort'):
    if len(g) >= 3:
        ax.plot(g.age, g.logc, 'o-', label=f"born {int(c)}-{int(c)+4}")
ax.set_xlabel('age of head')
ax.set_ylabel('mean log food consumption')
ax.legend(frameon=False, fontsize=7, ncol=2)
plt.show()

Each line is one cohort tracked across rounds.  Where the lines lie on top of
one another, there is no cohort effect and the shape is lifecycle.  Where
they are shifted vertically, later cohorts are better off at every age — a
cohort effect.  Where *every* line jumps in the same round, that is a period
effect, or a change in the questionnaire.



## The Uganda panel



In [1]:
uga = ll.Country('Uganda')
print(uga.waves)
print(uga.data_scheme)

In [1]:
import statsmodels.api as sm

ufood = uga.food_expenditures()
uc = ufood.groupby(['t', 'i']).sum().squeeze()
uc = np.log(uc.where(uc > 0)).rename('logc')

un = uga.household_characteristics().sum(axis=1).groupby(['t', 'i']).first()
un = np.log(un.astype(float).where(un > 0)).rename('logn')

p = pd.concat([uc, un], axis=1).dropna()
p = p[np.isfinite(p).all(axis=1)]

# How many households are actually observed more than once?
counts = p.groupby('i').size()
print(counts.value_counts().sort_index())

In [1]:
# Keep only households seen more than once -- singletons contribute nothing
# to a within estimator, and quietly inflate the apparent sample size.
p = p[counts.reindex(p.index.get_level_values('i')).to_numpy() > 1]

# Time dummies, then demean everything within household.  After demeaning
# there is no constant to include.
T = pd.get_dummies(p.index.get_level_values('t'), prefix='t',
                   drop_first=True, dtype=float)
T.index = p.index
q = pd.concat([p, T], axis=1)
qd = q - q.groupby(level='i').transform('mean')

res = sm.OLS(qd.logc, qd.drop(columns='logc')).fit(
    cov_type='cluster', cov_kwds={'groups': qd.index.get_level_values('i')})
print(res.summary().tables[1])

Compare the coefficient on `logn` here with the cross-sectional one from
session 4.  If they differ, the cross-sectional estimate was contaminated by
whatever it is about large households that also predicts consumption.



## The same panel, in welfare units



Everything above treats $\log c$ as the outcome.  Session 4 argued that
$w = -\log\lambda$ is the better welfare measure, because prices and
household composition are swept into the good-time effects and the Barten
scales rather than left in the number.  Here is the same regression on
$w$, so you can see how much it matters.



In [1]:
from pathlib import Path
import lsms_library as ll
import cfe
from cfe import Regression
import numpy as np, pandas as pd

def uganda_cfe():
    # The estimated CFE system for Uganda, as a cfe.Regression.  The object
    # carries beta, gamma, w = -log lambda, and predicted expenditures, so
    # nothing downstream has to refit.  Uses a copy staged on the hub if
    # there is one, else a copy you estimated earlier, else estimates it
    # from scratch (about forty seconds) and caches the result.  So this
    # cell is self-contained: no other notebook need have been run first.
    staged = Path('/srv/data/hhsurveys/uganda.rgsn')
    mine   = Path.home() / '.cache' / 'hhsurveys' / 'uganda.rgsn'
    for c in (staged, mine):
        if c.exists():
            return cfe.read_pickle(str(c))

    uga = ll.Country('Uganda')
    x = uga.food_expenditures().squeeze()
    agg = (uga.categorical_mapping['harmonize_food']
              .set_index('Preferred Label')['Aggregate Label'].to_dict())
    x = x.rename(index=agg, level='j')
    x = x.groupby(x.index.names).sum()

    y = np.log(x.replace(0, np.nan).dropna()).groupby(['i', 't', 'j']).sum()
    y = pd.concat({1: y}, names=['m']).reorder_levels(['i', 't', 'm', 'j']).sort_index()

    d0 = uga.household_characteristics()
    d = d0.assign(
        Girls=d0[[f'F {a}' for a in ['00-03', '04-08', '09-13', '14-18']]].sum(axis=1),
        Boys =d0[[f'M {a}' for a in ['00-03', '04-08', '09-13', '14-18']]].sum(axis=1),
        Women=d0[[f'F {a}' for a in ['19-30', '31-50', '51+']]].sum(axis=1),
        Men  =d0[[f'M {a}' for a in ['19-30', '31-50', '51+']]].sum(axis=1),
    )[['Girls', 'Boys', 'Women', 'Men', 'log HSize']].dropna(how='any')
    d = d.groupby(['i', 't']).first()
    d = pd.concat({1: d}, names=['m']).reorder_levels(['i', 't', 'm']).sort_index()

    r = Regression(y=y, d=d)
    r.get_beta(); r.get_w(); r.predicted_expenditures()
    mine.parent.mkdir(parents=True, exist_ok=True)
    r.to_pickle(str(mine))
    return r

r = uganda_cfe()
w = r.get_w()
w.groupby('t').agg(['size', 'mean', 'std']).round(3)

In [1]:
# The same within estimator, in welfare units.
# w arrives indexed (i, t, m) while p is (t, i).  Line them up explicitly:
# concat on a differently-ordered MultiIndex does not raise, it just fails
# to align -- the same trap that emptied the pseudo-panel above.
wi = w.droplevel('m') if 'm' in (w.index.names or []) else w
wi = wi.rename('w').reorder_levels(['t', 'i']).sort_index()

q2 = pd.concat([wi, p.logn], axis=1).dropna()
assert len(q2) > 0.5 * min(len(wi), len(p)), "w and p failed to align"

c2 = q2.groupby('i').size()
q2 = q2[c2.reindex(q2.index.get_level_values('i')).to_numpy() > 1]

T2 = pd.get_dummies(q2.index.get_level_values('t'), prefix='t',
                    drop_first=True, dtype=float)
T2.index = q2.index
Q2 = pd.concat([q2, T2], axis=1)
Q2d = Q2 - Q2.groupby(level='i').transform('mean')

res_w = sm.OLS(Q2d.w, Q2d.drop(columns='w')).fit(
    cov_type='cluster', cov_kwds={'groups': Q2d.index.get_level_values('i')})
print(res_w.summary().tables[1])

Three comparisons to make, and the third is the one to remember.

The coefficient on `logn`.  In $\log c$ it is about $+0.34$: a bigger
household simply spends more.  In $w$ it is about $-0.03$ — small and
*negative*.  Household size has already been absorbed into the Barten
scales, so what is left is what size does to welfare net of need, and that
is slightly adverse.  The two numbers are not rival estimates of one
parameter; they answer different questions.

The time effects.  In $\log c$ they climb steeply and monotonically, from
0.40 to 1.20 across the panel.  That is mostly inflation, since nothing
deflated those expenditures.  In $w$ they sit within $\pm 0.15$ and do
not trend, because prices live in $a^j_t$ and never entered $w$ at
all.  No CPI was used to achieve that, and none was available.

Now look at 2009–10 in the $w$ column.  It is the only negative time
effect in the panel.  That is the 2008 food price crisis, and it is
invisible in $\log c$, where 2009–10 is simply another step up the
inflation ladder.  Session 6 takes that observation and asks what it does to
the poverty rate.



## Exercises



1.  Rebuild the Ghana pseudo-panel with ten-year cohorts.  How much do the
    lifecycle profiles change?  Which conclusion is robust?
2.  Implement the Verbeek–Nijman errors-in-variables correction using the
    within-cell variances, and report the corrected $\beta$.
3.  In Uganda, quantify attrition: what fraction of the 2009–10 households
    appear in 2011–12?  Are the leavers different in 2009–10 consumption
    from the stayers?  What does that do to your fixed-effects estimate?

